# 01 · Exploration des données (EDA)

**Projet :** Pilotage des risques de chantier · Arthy Aroul & Aurélie Mahé · Bootcamp Data Essentials, Jedha (2026)

**Question du projet :** peut-on prédire le niveau de risque d'un chantier à partir de ses indicateurs opérationnels et environnementaux ?

**Objectif de ce notebook :** comprendre le dataset avant toute modélisation (qualité, cohérence, distribution de la cible, liens entre variables et risque) et repérer les pièges, notamment les fuites de données.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## 1. Chargement des données

In [ ]:
# Chemin du fichier (voir data/README.md pour télécharger le dataset)
# Sur Google Colab : téléverser le CSV puis utiliser "/content/bim_ai_civil_engineering_dataset.csv"
DATA_PATH = "../data/bim_ai_civil_engineering_dataset.csv"

df = pd.read_csv(DATA_PATH)
print(f"{df.shape[0]} projets, {df.shape[1]} variables")

df["Start_Date"] = pd.to_datetime(df["Start_Date"])
df["End_Date"] = pd.to_datetime(df["End_Date"])
df.head()

## 2. Qualité des données

In [ ]:
df.info()

In [ ]:
print("Valeurs manquantes :", df.isnull().sum().sum())
print("Lignes dupliquées  :", df.duplicated().sum())
print("Projets uniques    :", df["Project_ID"].nunique(), "/", len(df))

Le dataset est complet : aucune valeur manquante, aucun doublon, un identifiant unique par projet. Il ne demande pas de nettoyage, ce qui est déjà inhabituel pour des données réelles.

### Cohérence des variables calculées

On vérifie que les dépassements correspondent bien à la différence entre réel et prévu.

In [ ]:
ecart_cout = (df["Cost_Overrun"] - (df["Actual_Cost"] - df["Planned_Cost"])).abs().max()
ecart_delai = (df["Schedule_Deviation"] - (df["Actual_Duration"] - df["Planned_Duration"])).abs().max()

print(f"Écart maximal Cost_Overrun vs (réel - prévu)      : {ecart_cout:.1e}")
print(f"Écart maximal Schedule_Deviation vs (réel - prévu) : {ecart_delai:.1e}")

Les écarts sont de l'ordre de l'arrondi informatique : `Cost_Overrun` et `Schedule_Deviation` sont exactement « réel − prévu ». Ce sont des **informations redondantes**, à garder en tête pour la modélisation.

## 3. Statistiques descriptives par famille de variables

In [ ]:
familles = {
    "Budget": ["Planned_Cost", "Actual_Cost", "Cost_Overrun"],
    "Délais": ["Planned_Duration", "Actual_Duration", "Schedule_Deviation"],
    "Sécurité": ["Accident_Count", "Safety_Risk_Score"],
    "Ressources": ["Energy_Consumption", "Material_Usage", "Labor_Hours", "Equipment_Utilization"],
    "Conditions": ["Temperature", "Humidity", "Air_Quality_Index", "Vibration_Level"],
}

for nom, colonnes in familles.items():
    print(f"\n--- {nom} ---")
    display(df[colonnes].describe().round(2))

In [ ]:
for column in ["Project_Type", "Location", "Weather_Condition", "Risk_Level", "Anomaly_Detected"]:
    print(f"\n--- {column} ---")
    print(df[column].value_counts())

Les modalités de chaque variable catégorielle sont presque parfaitement équilibrées (environ 200 projets par type d'ouvrage, par ville et par météo). Dans la réalité, on n'aurait pas autant de barrages que de routes : c'est un premier indice que les données sont **synthétiques**.

## 4. La variable cible : `Risk_Level`

In [ ]:
ordre_risque = ["Low", "Medium", "High"]

print(df["Risk_Level"].value_counts(normalize=True).reindex(ordre_risque).round(3))

sns.countplot(data=df, x="Risk_Level", order=ordre_risque, hue="Risk_Level", legend=False)
plt.title("Répartition des niveaux de risque")
plt.xlabel("Niveau de risque")
plt.ylabel("Nombre de projets")
plt.show()

Les classes sont **déséquilibrées** : environ 50 % High, 34 % Medium et 16 % Low. Il faudra en tenir compte dans le choix des métriques.

In [ ]:
display(df.groupby("Risk_Level")["Safety_Risk_Score"].describe().reindex(ordre_risque).round(2))

sns.boxplot(data=df, x="Risk_Level", y="Safety_Risk_Score", order=ordre_risque)
plt.title("Safety_Risk_Score selon le niveau de risque")
plt.show()

`Risk_Level` est très fortement déterminé par `Safety_Risk_Score` : les projets Low ont un score toujours inférieur à 3, les High sont concentrés en haut de l'échelle. Garder ce score comme variable explicative reviendrait à **donner la réponse au modèle** (fuite de données). Il sera donc retiré de la modélisation.

In [ ]:
pd.crosstab(df["Risk_Level"], df["Anomaly_Detected"], normalize="index").reindex(ordre_risque).round(3)

Aucun projet Low n'a d'anomalie détectée. `Anomaly_Detected` pourrait donc être liée à la façon dont le risque est calculé : c'est une variable à surveiller.

## 5. Distribution des variables numériques

In [ ]:
colonnes_num = df.select_dtypes(include="number").columns.drop("Anomaly_Detected")

df[colonnes_num].hist(bins=30, figsize=(16, 12))
plt.suptitle("Distribution des variables numériques", y=1.0)
plt.tight_layout()
plt.show()

La plupart des histogrammes sont **plats** : les valeurs sont réparties uniformément entre un minimum et un maximum. Des données réelles montreraient plutôt des distributions asymétriques (beaucoup de petits chantiers, quelques très gros).

On peut le vérifier : pour une loi uniforme, l'écart-type vaut (max − min) / √12. Un ratio proche de 1 indique une distribution compatible avec une loi uniforme.

In [ ]:
ratio_uniforme = (
    df[colonnes_num].std() / ((df[colonnes_num].max() - df[colonnes_num].min()) / np.sqrt(12))
).sort_values()

ratio_uniforme.round(2).to_frame("écart-type / écart-type uniforme")

Pour la grande majorité des variables, le ratio est proche de 1 : les données ont très probablement été **générées aléatoirement**, variable par variable. C'est la limite principale du dataset.

## 6. Prévu vs réel

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=df, x="Planned_Cost", y="Actual_Cost", ax=axes[0], s=15)
axes[0].set_title("Coût prévu vs coût réel")
sns.scatterplot(data=df, x="Planned_Duration", y="Actual_Duration", ax=axes[1], s=15)
axes[1].set_title("Durée prévue vs durée réelle")
plt.tight_layout()
plt.show()

print("Corrélation coût prévu / réel   :", round(df["Planned_Cost"].corr(df["Actual_Cost"]), 3))
print("Corrélation durée prévue / réelle :", round(df["Planned_Duration"].corr(df["Actual_Duration"]), 3))

Le réel suit de près le prévu (corrélations supérieures à 0,9), avec un dépassement moyen d'environ 5 M et de 107 jours. Les projets dépassent plus souvent qu'ils ne restent sous le budget.

## 7. Liens entre les variables et le risque

In [ ]:
variables_explicatives = colonnes_num.drop(["Safety_Risk_Score"])
correlations = df[variables_explicatives].corrwith(df["Safety_Risk_Score"]).sort_values()

correlations.plot(kind="barh", figsize=(7, 6))
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Corrélation de chaque variable avec Safety_Risk_Score")
plt.xlabel("Coefficient de corrélation (Pearson)")
plt.tight_layout()
plt.show()

print("Corrélation maximale en valeur absolue :", round(correlations.abs().max(), 3))

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(df[colonnes_num].corr(), cmap="coolwarm", center=0)
plt.title("Matrice de corrélation des variables numériques")
plt.show()

En dehors des paires évidentes (prévu / réel / dépassement), la matrice est quasiment vide : **aucune variable n'est corrélée au score de risque** (corrélation maximale de l'ordre de 0,07).

### Axe 1 · Planning et pression temporelle

*Les chantiers en avance sont-ils aussi ceux qui ont le moins d'accidents ?*

In [ ]:
df["Quartile_retard"] = pd.qcut(
    df["Schedule_Deviation"], 4,
    labels=["Q1 (avance ou faible retard)", "Q2", "Q3", "Q4 (fort retard)"],
)

sns.boxplot(data=df, x="Quartile_retard", y="Accident_Count")
plt.title("Nombre d'accidents selon le quartile de retard")
plt.xlabel("Quartile de Schedule_Deviation")
plt.xticks(rotation=15)
plt.show()

df.groupby("Quartile_retard", observed=True)["Accident_Count"].mean().round(2)

### Axe 2 · Effectifs

*Existe-t-il un seuil de main-d'œuvre au-delà duquel le risque n'augmente plus ?*

In [ ]:
df["Tranche_heures"] = pd.cut(df["Labor_Hours"], bins=8)
moyenne_par_tranche = df.groupby("Tranche_heures", observed=True)["Safety_Risk_Score"].mean()

moyenne_par_tranche.plot(marker="o", figsize=(9, 4))
plt.title("Safety_Risk_Score moyen selon les heures de travail (8 tranches)")
plt.xlabel("Tranche de Labor_Hours")
plt.ylabel("Score de risque moyen")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### Axe 3 · Dérives budgétaires et climat

*Le dépassement budgétaire est-il plus lié au planning qu'au climat ?*

In [ ]:
variables_axe3 = ["Schedule_Deviation", "Actual_Duration", "Temperature", "Humidity", "Air_Quality_Index"]
df[variables_axe3].corrwith(df["Cost_Overrun"]).round(3).to_frame("Corrélation avec Cost_Overrun")

Sur ces trois axes, les écarts observés restent faibles et sans tendance claire, ce qui est cohérent avec l'absence de corrélation vue plus haut.

### Focus · Consommation énergétique

Dans la réalité, un barrage ou un tunnel consomme beaucoup plus d'énergie qu'une route. Voyons si le dataset reflète ces différences.

In [ ]:
display(df.groupby("Project_Type")["Energy_Consumption"].agg(["mean", "median"]).round(0))

sns.boxplot(data=df, x="Project_Type", y="Energy_Consumption")
plt.title("Consommation énergétique par type d'ouvrage")
plt.show()

df[["Material_Usage", "Labor_Hours", "Equipment_Utilization", "Planned_Cost"]].corrwith(
    df["Energy_Consumption"]
).round(3).to_frame("Corrélation avec Energy_Consumption")

La consommation d'énergie ne dépend ni du type d'ouvrage, ni des heures de travail, ni des matériaux, ni du budget. Sur des données réelles, ces liens seraient forts : c'est une confirmation supplémentaire du caractère synthétique du dataset.

## 8. Conclusions de l'EDA

1. **Données propres mais synthétiques** : aucune valeur manquante, catégories quasi équilibrées et distributions uniformes. Les liens attendus dans la réalité (type d'ouvrage et énergie, par exemple) sont absents.
2. **Variables redondantes** : `Cost_Overrun` et `Schedule_Deviation` sont exactement « réel − prévu ».
3. **Fuite de données** : `Risk_Level` est dérivé de `Safety_Risk_Score`, qui doit être exclu des variables explicatives. `Anomaly_Detected` est à surveiller.
4. **Signal très faible** : aucune variable n'est corrélée au risque (maximum de l'ordre de 0,07). Il faut s'attendre à des modèles peu performants, et donc les comparer à une baseline naïve.
5. **Cible déséquilibrée** : les métriques devront en tenir compte (F1 macro et surtout MCC plutôt que l'accuracy).